# Colab Model Station

An automated, reproducible deployment framework for Google Colab GPU runtimes.
Supports multiple inference backends: **LLMs (vLLM / Ollama)** and **Diffusion (FastAPI + Diffusers / ComfyUI)** with dynamic **FLUX**, **SDXL**, and **LoRA** switching over **Tailscale Mesh**, **Cloudflare Tunnels**, or **VS Code Remote Tunnels**.

---

## Step 0: Repository Setup & Hardware Inspection
Clones `colab-inference-node` to local runtime storage (`/content/colab-inference-node`), injects Colab Secrets, and displays GPU allocation.

In [ ]:
# ==============================================================================
# Step 0: Clone Repository, Inject Secrets, and Inspect GPU
# ==============================================================================
import os
from google.colab import userdata

# 1. Inject Secrets from Colab Secrets store (if configured)
try:
    os.environ['TAILSCALE_AUTHKEY'] = userdata.get('TAILSCALE_AUTHKEY')
except Exception:
    pass

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. Clone or update repository
!git clone https://github.com/yixuanw99/colab-inference-node.git /content/colab-inference-node 2>/dev/null || (cd /content/colab-inference-node && git pull)

# 3. Hardware Profile & Model Catalog
!cd /content/colab-inference-node && bash engine.sh status
!cd /content/colab-inference-node && bash engine.sh list


## Step 1: Environment Provisioning

Choose and install dependencies based on your target workload:
- **Diffusion Engine** (`bash setup.sh diffusers`): FastAPI, Diffusers, PEFT, Accelerate.
- **Ollama Engine** (`bash setup.sh ollama`): Standalone lightweight GGUF runtime (~15s install).
- **vLLM Engine** (`bash setup.sh vllm`): High-throughput PagedAttention inference server.
- **ComfyUI Engine** (`bash setup.sh comfyui`): Clones ComfyUI core and dependencies.
- **Network Tunnels** (`bash setup.sh tunnels`): Tailscale, Cloudflared, and VS Code CLI.
- **Full Stack** (`bash setup.sh all`): Installs all base utilities.

In [ ]:
%%bash
cd /content/colab-inference-node

# Select provisioning target:
# Option A: Diffusion models (FLUX / SDXL)
bash setup.sh diffusers

# Option B: Ollama LLMs (T4 / L4)
# bash setup.sh ollama

# Option C: vLLM Enterprise LLMs (A100 / L4)
# bash setup.sh vllm


## Hardware & Model Allocation Matrix

| GPU Tier | VRAM | Engine | Target Model | Precision / Format | Offload Mode | Workload |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **NVIDIA T4** | 15 GB | **Ollama** | `qwen2.5-coder:7b` | GGUF Q4_K_M | Full VRAM | Lightweight code assistance |
| **NVIDIA T4** | 15 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | Sequential Offload | 4-step diffusion generation |
| **NVIDIA T4** | 15 GB | **Diffusers** | `stabilityai/stable-diffusion-xl-base-1.0` | FP16 | Model Offload | SDXL latent diffusion |
| **NVIDIA L4** | 24 GB | **Ollama** | `qwen2.5-coder:32b` | GGUF Q4_K_M | Full VRAM | Advanced IDE coding |
| **NVIDIA L4** | 24 GB | **vLLM** | `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ` | 4-bit AWQ | Full VRAM | High-throughput concurrent serving |
| **NVIDIA L4** | 24 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | Model Offload | Fast FLUX generation (~15s) |
| **NVIDIA A100** | 40/80 GB | **vLLM** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` | bfloat16 | Full VRAM | Unquantized reasoning serving |
| **NVIDIA A100** | 40/80 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | BF16 | Full VRAM | Native unquantized FLUX (~5s) |

---

## Deployment Solution A: Diffusion Serving (FLUX.1 / SDXL + LoRA) via Tailscale

Launches FastAPI Diffusers service on port 8000 and exposes it to your private Tailnet.

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. Connect to Tailscale Mesh in Userspace mode
bash engine.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. Launch FastAPI Diffusers inference server (FLUX.1-schnell in FP8)
bash engine.sh diffusers start --model "black-forest-labs/FLUX.1-schnell" --precision fp8

# 3. Proxy port 8000 to Tailnet
bash engine.sh tunnel tailscale serve 8000

# 4. Arm Idle Watchdog (30 minute compute unit protection)
bash engine.sh watchdog start --port 8000 --timeout 1800

echo "================================================================"
echo "Diffusers Endpoint: http://colab-model-station:8000"
echo "================================================================"


## Deployment Solution B: LLM Serving (vLLM / Ollama) via Tailscale

Launches an OpenAI-compatible text generation service for OpenCode, Cursor, or Aider.

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. Ensure Tailscale is active
bash engine.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. Choose LLM Engine:
# Option B1: Ollama on Port 11434
bash engine.sh ollama start
bash engine.sh ollama pull qwen2.5-coder:7b
bash engine.sh tunnel tailscale serve 11434

# Option B2: vLLM on Port 8000 (A100 / L4)
# bash engine.sh vllm start --model "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
# bash engine.sh tunnel tailscale serve 8000


## Deployment Solution C: VS Code Remote Tunnel (Web & Desktop IDE)

Enables zero-configuration remote development via `https://vscode.dev/tunnel/<name>`. Credentials are saved to Google Drive if mounted.

In [ ]:
%%bash
cd /content/colab-inference-node

# Setup, authenticate GitHub, and launch VS Code Tunnel
bash engine.sh tunnel vscode setup
bash engine.sh tunnel vscode login
bash engine.sh tunnel vscode start "colab-model-station"


## Deployment Solution D: ComfyUI Web GUI (Node-Based Diffusion)

Launches ComfyUI server on port 8188 and provides direct web access via Cloudflare Tunnel, Tailscale, or native Colab port proxy.

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. Start ComfyUI daemon on port 8188
bash engine.sh comfyui start

# 2. Expose via Cloudflare Public Tunnel (instant public HTTPS link)
bash engine.sh tunnel cloudflare up 8188

# 3. Proxy port 8188 to Tailscale (if Tailnet is connected)
bash engine.sh tunnel tailscale serve 8188


In [ ]:
# Alternative: Native Google Colab In-Notebook Browser Link
try:
    from google.colab.output import serve_kernel_port_as_window
    serve_kernel_port_as_window(8188, anchor_text="Open ComfyUI Web Interface (Colab Proxy)")
except ImportError:
    print("Colab port proxy only available inside Google Colab.")


## Step 2: In-Notebook Verification

Test live inference response directly within this notebook.

In [ ]:
import requests
import base64
from IPython.display import display, Image

# Test Diffusion Output
try:
    res = requests.post(
        "http://127.0.0.1:8000/v1/images/generations",
        json={"prompt": "A futuristic laboratory on Mars, hyperrealistic", "num_inference_steps": 4, "return_base64": True},
        timeout=60
    ).json()
    print(f"Diffusion Generation Success ({res['generation_time_sec']}s)")
    display(Image(base64.b64decode(res['images'][0]['base64_data'])))
except Exception as e:
    print(f"Diffusion test note: {e}")


## Step 3: Compute Unit Protection & Runtime Teardown

> **CRITICAL**: Closing your browser tab does **NOT** stop your Colab GPU instance. Paid compute units will continue to drain until hard timeout.

Run this final cell to release all GPU VRAM and terminate the Colab instance immediately.

In [ ]:
# ==============================================================================
# Step 3: Terminate Colab Runtime and Stop Billing
# ==============================================================================
from google.colab import runtime
print("Disconnecting Google Colab runtime and releasing compute units...")
runtime.unassign()
